# Google Colab R

- Google is now supporting a Colab notebook with the R kernel
- We can run R codes directly on Colab
- R Colab Notebook Link: [R Google Colab](https://colab.fan/r)
    - Colab with R kernel
    - With base-R installed
    - Run R codes immediately
    - No need to setup
    - Save a copy in your Drive


In [3]:
################################################################################
# 00_setup_municipios.R (VERSÃO CORRIGIDA E REVISADA)
#
# OBJETIVO:
#   - Preparar a estrutura de pastas do projeto.
#   - Baixar e padronizar a base-mestre dos 417 municípios da Bahia (UF = 29)
#     diretamente da API do IBGE, criando os códigos de 7 e 6 dígitos.
################################################################################

## 1. Instalação e Carregamento dos Pacotes Essenciais ------------------------

pacotes <- c("dplyr", "stringr", "jsonlite", "readr", "janitor")

instalar_faltantes <- function(pkgs) {
  faltantes <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(faltantes) > 0) {
    install.packages(faltantes, dependencies = TRUE)
  }
}

instalar_faltantes(pacotes)
invisible(lapply(pacotes, library, character.only = TRUE))

## 2. Estrutura de Pastas do Projeto -----------------------------------------

# Garante a criação dos diretórios locais sem depender do pacote 'here'
diretorios <- c("data/raw", "data/processed", "data/cache")
lapply(diretorios, function(dir) {
  if (!dir.exists(dir)) dir.create(dir, recursive = TRUE, showWarnings = FALSE)
})

message(">> Diretórios de dados verificados e criados com sucesso.")

## 3. Importação da Lista Oficial dos 417 Municípios da Bahia -----------------

url_ibge <- "https://servicodados.ibge.gov.br/api/v1/localidades/estados/29/municipios"

message(">> Conectando à API do IBGE...")

municipios_ba <- tryCatch({
  # Leitura direta via jsonlite (mais robusta e compatível)
  dados_raw <- jsonlite::fromJSON(url_ibge)

  # Tratamento e padronização dos códigos IBGE (7 e 6 dígitos)
  dados_proc <- dados_raw %>%
    janitor::clean_names() %>%
    transmute(
      cod_ibge7      = str_pad(as.character(id), width = 7, side = "left", pad = "0"),
      cod_ibge6      = str_sub(cod_ibge7, 1, 6),
      nome_municipio = nome
    ) %>%
    arrange(nome_municipio)

  dados_proc
}, error = function(e) {
  message(">> ERRO ao conectar à API do IBGE: ", e$message)
  return(NULL)
})

## 4. Validação e Salvamento da Base-Mestre -----------------------------------

if (!is.null(municipios_ba)) {
  n_mun <- nrow(municipios_ba)

  if (n_mun == 417) {
    message(">> SUCESSO: Todos os 417 municípios da Bahia foram carregados.")
  } else {
    warning(sprintf(">> ATENÇÃO: A API retornou %d municípios (esperado: 417).", n_mun))
  }

  # Salvar nos formatos RDS (preserva tipos no R) e CSV
  saveRDS(municipios_ba, "data/processed/municipios_ba.rds")
  write_csv(municipios_ba, "data/processed/municipios_ba.csv")

  message(">> Base-mestre salva em:")
  message("   - data/processed/municipios_ba.rds")
  message("   - data/processed/municipios_ba.csv")

} else {
  stop("A base de municípios não pôde ser gerada. Verifique a sua conexão de internet/proxy.")
}

## 5. Parâmetros Globais do Projeto -------------------------------------------

params <- list(
  ano_inicio  = 2015,
  ano_fim     = 2024,
  uf_sigla    = "BA",
  uf_cod_ibge = "29",
  portal_transparencia_key = Sys.getenv("PORTAL_TRANSPARENCIA_API_KEY")
)

saveRDS(params, "data/processed/params_projeto.rds")

if (params$portal_transparencia_key == "") {
  message(">> AVISO: 'PORTAL_TRANSPARENCIA_API_KEY' não foi encontrada nas variáveis de ambiente.")
  message("   Configure no .Renviron caso necessite realizar requisições automáticas via API.")
}

message("\n>> Setup concluído com sucesso!")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘proxy’, ‘e1071’, ‘wk’, ‘classInt’, ‘s2’, ‘units’, ‘igraph’, ‘snakecase’, ‘RSQLite’, ‘sf’, ‘tidygraph’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test




[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

>> Diretórios de dados verificados e criados com sucesso.

>> Conectando à API do IBGE...

>> SUCESSO: Todos os 417 municípios da Bahia foram carregados.

>> Base-mestre salva em:

   - data/processed/municipios_ba.rds

   - data/processed/municipios_ba.csv


>> Setup concluído com sucesso!



In [9]:
################################################################################
# 05_controles_demograficos_ibge_sei.R (VERSÃO CORRIGIDA E REVISADA)
#
# OBJETIVO:
#   (A) Extrair dados de População, PIB municipal e Área do IBGE/SIDRA.
#   (B) Mapear a dummy regional do Território de Identidade Baixo Sul (SEI-BA).
#   (C) Construir as variáveis: populacao, log_pib_per_capita, densidade_demografica.
################################################################################

## 1. Pacotes Essenciais ------------------------------------------------------
pacotes <- c("dplyr", "stringr", "readr", "janitor", "purrr", "tidyr", "zoo", "sidrar")
# ^ "sidrar" estava faltando aqui - é usado logo abaixo (seção de população)
#   sem nenhuma checagem antes, diferente das seções de PIB/área que ao menos
#   verificam com tem_sidrar. Por isso o erro "there is no package called
#   'sidrar'" só apareceu disfarçado de falha de rede, repetido 3x por ano.

instalar_faltantes <- function(pkgs) {
  faltantes <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(faltantes) > 0) install.packages(faltantes, dependencies = TRUE)
}
instalar_faltantes(pacotes)
invisible(lapply(pacotes, library, character.only = TRUE))

tem_sidrar <- requireNamespace("sidrar", quietly = TRUE)

## 2. Carregar Base-Mestre de Municípios e Parâmetros -------------------------
caminho_mestre <- "data/processed/municipios_ba.rds"
if (file.exists(caminho_mestre)) {
  municipios_ba <- readRDS(caminho_mestre)
} else {
  stop("Base-mestre de municípios não encontrada. Execute o Script 00 primeiro!")
}

params <- if (file.exists("data/processed/params_projeto.rds")) {
  readRDS("data/processed/params_projeto.rds")
} else {
  list(ano_inicio = 2015, ano_fim = 2024)
}

anos_painel <- seq(params$ano_inicio, params$ano_fim)

# Função para remover acentos
remover_acentos <- function(texto) {
  iconv(toupper(trimws(texto)), to = "ASCII//TRANSLIT")
}

## 3. (A1) POPULAÇÃO RESIDENTE ESTIMADA - CRIAÇÃO E SALVAMENTO DE populacao_ba -----
message(">> Processando população residente do IBGE (2015-2024)...")

## Busca 1 ano da Tabela 6579 do SIDRA, com retry em caso de falha de rede.
## Ao contrário da versão anterior (try(silent = TRUE)), aqui o erro real é
## sempre mostrado - um try() silencioso escondia que TODAS as chamadas
## estavam falhando, e isso só aparecia muito depois, num left_join()
## totalmente alheio ("Join columns in y must be present in the data"),
## sem nenhuma pista do que causou.
baixar_populacao_ano <- function(a, max_tentativas = 3) {
  for (tentativa in seq_len(max_tentativas)) {
    res <- tryCatch(
      sidrar::get_sidra(x = 6579, period = as.character(a), geo = "City",
                        geo.filter = list("State" = 29)),
      error = function(e) {
        message(sprintf("  [ano %s, tentativa %d/%d] falhou: %s",
                        a, tentativa, max_tentativas, conditionMessage(e)))
        NULL
      }
    )
    if (!is.null(res) && nrow(res) > 0) return(res)
    if (tentativa < max_tentativas) Sys.sleep(5 * tentativa)  # 5s, 10s...
  }
  NULL
}

pop_list <- list()
for (a in anos_painel) {
  res <- baixar_populacao_ano(a)
  if (!is.null(res)) {
    pop_list[[as.character(a)]] <- res %>%
      janitor::clean_names() %>%
      transmute(
        cod_ibge6 = str_sub(as.character(municipio_codigo), 1, 6),
        ano       = as.numeric(ano),
        populacao = as.numeric(valor)
      )
  } else {
    message(sprintf(">> ATENÇÃO: nenhum dado de população obtido para o ano %s ", a),
            "após todas as tentativas.")
  }
}

populacao_raw <- bind_rows(pop_list)

# Falha rápido e com mensagem clara, em vez de deixar o problema estourar
# silenciosamente lá na frente, num left_join() sem nenhuma relação aparente
# com a causa real (foi exatamente o que aconteceu antes dessa correção).
if (nrow(populacao_raw) == 0) {
  stop("Nenhum dado de população foi obtido do SIDRA para NENHUM ano - ",
       "a Tabela 6579 pode ter mudado de estrutura, ou a API do SIDRA/IBGE ",
       "está inacessível agora. Veja as mensagens de erro acima (por ano) ",
       "para a causa exata antes de prosseguir.")
}

# Painel base garantindo os 417 municípios x 10 anos
painel_pop <- tidyr::expand_grid(
  cod_ibge6 = as.character(municipios_ba$cod_ibge6),
  ano       = anos_painel
)

populacao_ba <- painel_pop %>%
  left_join(populacao_raw, by = c("cod_ibge6", "ano")) %>%
  arrange(cod_ibge6, ano) %>%
  group_by(cod_ibge6) %>%
  # Imputação (LOCF) para preencher eventuais anos ausentes na API (ex: 2021)
  mutate(
    populacao = zoo::na.locf(populacao, na.rm = FALSE),
    populacao = zoo::na.locf(populacao, fromLast = TRUE, na.rm = FALSE)
  ) %>%
  ungroup() %>%
  left_join(municipios_ba %>% select(cod_ibge6, cod_ibge7, nome_municipio), by = "cod_ibge6")

# SALVAMENTO EXPLICITO DO ARQUIVO DE POPULAÇÃO (Exigido pelo Script 01)
saveRDS(populacao_ba, "data/processed/populacao_ba.rds")
write_csv(populacao_ba, "data/processed/populacao_ba.csv")

message(">> SUCESSO: Arquivo 'data/processed/populacao_ba.rds' criado com sucesso!")
## 4. (A2) PIB MUNICIPAL E ÁREA TERRITORIAL ----------------------------------
arq_pib_salvo <- "data/raw/pib_siconfi_raw.rds"

if (tem_sidrar) {
  message(">> Consultando API do SIDRA para PIB Municipal...")
  pib_list <- list()
  for (a in anos_painel) {
    try({
      res_pib <- sidrar::get_sidra(x = 5938, period = as.character(a), geo = "City", geo.filter = list("State" = 29), variable = 37)
      if (!is.null(res_pib) && nrow(res_pib) > 0) {
        pib_list[[as.character(a)]] <- res_pib %>%
          janitor::clean_names() %>%
          transmute(
            cod_ibge6    = str_sub(as.character(municipio_codigo), 1, 6),
            ano          = as.numeric(ano),
            pib_corrente = as.numeric(valor) * 1000 # Convertendo mil R$ para R$
          )
      }
    }, silent = TRUE)
  }
  pib_ba <- bind_rows(pib_list)
} else {
  pib_ba <- tibble(cod_ibge6 = character(), ano = numeric(), pib_corrente = numeric())
}

# Área Territorial (Tabela 1301 ou constante municipal)
if (tem_sidrar) {
  message(">> Consultando Área Territorial dos Municípios...")
  area_ba <- tryCatch({
    sidrar::get_sidra(x = 1301, geo = "City", geo.filter = list("State" = 29)) %>%
      janitor::clean_names() %>%
      transmute(
        cod_ibge6 = str_sub(as.character(municipio_codigo), 1, 6),
        area_km2  = as.numeric(valor)
      ) %>%
      distinct(cod_ibge6, .keep_all = TRUE)
  }, error = function(e) tibble(cod_ibge6 = character(), area_km2 = numeric()))
} else {
  area_ba <- tibble(cod_ibge6 = character(), area_km2 = numeric())
}

## 5. (B) DUMMY DO TERRITÓRIO DE IDENTIDADE BAIXO SUL (15 MUNICÍPIOS) ---------
# Mapeamento rigoroso via Códigos IBGE de 6 dígitos dos 15 municípios do Baixo Sul:
codigos_baixo_sul <- c(
  "290230", # Aratuípe
  "290540", # Cairu
  "290580", # Camamu
  "291120", # Gandu
  "291270", # Ibirapitanga
  "291345", # Igrapiúna
  "291730", # Ituberá
  "291780", # Jaguaripe
  "292260", # Nilo Peçanha
  "292467", # Piraí do Norte
  "292575", # Presidente Tancredo Neves
  "293120", # Taperoá
  "293160", # Teolândia
  "293290", # Valença
  "293350"  # Wenceslau Guimarães
)

territorios_ba <- municipios_ba %>%
  mutate(
    BaixoSul = ifelse(cod_ibge6 %in% codigos_baixo_sul, 1L, 0L)
  ) %>%
  select(cod_ibge6, BaixoSul)

message(">> Municípios identificados no Baixo Sul: ", sum(territorios_ba$BaixoSul), " de 15 esperados.")

# VALIDAÇÃO: confere se os códigos hardcoded acima realmente correspondem aos
# nomes esperados na base oficial do IBGE (municipios_ba). Isso existe porque
# uma versão anterior desta lista tinha 7 códigos digitados errados (cada um
# apontando silenciosamente para outro município da Bahia, sem gerar nenhum
# erro - só descoberto por checagem manual). Se algum nome não bater aqui, o
# script para, em vez de deixar a dummy corrompida se propagar pelo painel.
nomes_baixo_sul_esperados <- c(
  "290230" = "Aratuípe",
  "290540" = "Cairu",
  "290580" = "Camamu",
  "291120" = "Gandu",
  "291270" = "Ibirapitanga",
  "291345" = "Igrapiúna",
  "291730" = "Ituberá",
  "291780" = "Jaguaripe",
  "292260" = "Nilo Peçanha",
  "292467" = "Piraí do Norte",
  "292575" = "Presidente Tancredo Neves",
  "293120" = "Taperoá",
  "293160" = "Teolândia",
  "293290" = "Valença",
  "293350" = "Wenceslau Guimarães"
)

conferencia_baixo_sul <- municipios_ba %>%
  filter(cod_ibge6 %in% names(nomes_baixo_sul_esperados)) %>%
  mutate(nome_esperado = nomes_baixo_sul_esperados[cod_ibge6]) %>%
  filter(remover_acentos(nome_municipio) != remover_acentos(nome_esperado))

if (nrow(conferencia_baixo_sul) > 0) {
  print(conferencia_baixo_sul %>% select(cod_ibge6, nome_municipio, nome_esperado))
  stop("Divergência entre os códigos hardcoded do Baixo Sul e os nomes reais ",
       "na base do IBGE (municipios_ba) - ver tabela acima. Corrija ",
       "codigos_baixo_sul antes de prosseguir.")
}

## 6. CONSOLIDAÇÃO DOS CONTROLES DEMOGRÁFICOS ---------------------------------
message(">> Consolidando o painel de controles demográficos...")

controles_demograficos <- populacao_ba %>%
  # 1. Seleciona apenas as colunas garantidas do objeto de população
  select(cod_ibge6, ano, populacao) %>%

  # 2. Une PIB, Área Territorial e a dummy do Baixo Sul
  left_join(pib_ba, by = c("cod_ibge6", "ano")) %>%
  left_join(area_ba, by = "cod_ibge6") %>%
  left_join(territorios_ba, by = "cod_ibge6") %>%

  # 3. Injeta cod_ibge7 e nome_municipio diretamente da base-mestre (municipios_ba)
  left_join(municipios_ba, by = "cod_ibge6") %>%

  # 4. Ajustes e imputação (LOCF) para PIB nos anos mais recentes
  arrange(cod_ibge6, ano) %>%
  group_by(cod_ibge6) %>%
  mutate(
    pib_corrente          = zoo::na.locf(pib_corrente, na.rm = FALSE),
    pib_per_capita        = ifelse(!is.na(populacao) & populacao > 0, pib_corrente / populacao, NA_real_),
    log_pib_per_capita    = ifelse(!is.na(pib_per_capita) & pib_per_capita > 0, log(pib_per_capita), NA_real_),
    densidade_demografica = ifelse(!is.na(area_km2) & area_km2 > 0, populacao / area_km2, NA_real_)
  ) %>%
  ungroup() %>%

  # 5. Seleção limpa e organizada das colunas finais
  select(
    cod_ibge6,
    cod_ibge7,
    nome_municipio,
    ano,
    populacao,
    pib_corrente,
    pib_per_capita,
    log_pib_per_capita,
    area_km2,
    densidade_demografica,
    BaixoSul
  )

## 7. SALVAR RESULTADOS --------------------------------------------------------
saveRDS(controles_demograficos, "data/processed/controles_demograficos_ba.rds")
write_csv(controles_demograficos, "data/processed/controles_demograficos_ba.csv")

message("\n>> CONCLUÍDO COM SUCESSO!")
message("   - Painel salvo em: data/processed/controles_demograficos_ba.rds")
message("   - Total de registros (2015-2024): ", nrow(controles_demograficos), " (esperado: 4.170)")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

>> Processando população residente do IBGE (2015-2024)...

>> ATENÇÃO: nenhum dado de população obtido para o ano 2022 após todas as tentativas.

>> ATENÇÃO: nenhum dado de população obtido para o ano 2023 após todas as tentativas.

>> SUCESSO: Arquivo 'data/processed/populacao_ba.rds' criado com sucesso!

>> Consultando API do SIDRA para PIB Municipal...

>> Consultando Área Territorial dos Municípios...

>> Municípios identificados no Baixo Sul: 15 de 15 esperados.

>> Consolidando o painel de controles demográficos...


>> CONCLUÍDO COM SUCESSO!

   - Painel salvo em: data/processed/controles_demograficos_ba.rds

   - Total de registros (2015-2024): 4170 (esperado: 4.170)



In [11]:
################################################################################
# 01_emendas_parlamentares.R
#
# OBJETIVO
#   Construir, para os 417 municípios da Bahia e para cada ano do painel,
#   as variáveis:
#     - EmendasPC   = (emendas individuais + bancada + especiais "Pix" pagas) / população
#     - PartEmendas = emendas pagas / Receita Total do município
#
# FONTES
#   (A) Portal da Transparência (CGU) - API REST oficial, requer chave de API
#       -> emendas parlamentares (individuais, de bancada) pagas por município
#   (B) Transferegov.br - dados abertos (CKAN) - transferências especiais
#       ("emendas Pix", Art. 166-A da CF/88) e convênios/transferências
#       voluntárias oriundas de emendas
#   (C) Siga Brasil (Senado Federal) - NÃO possui API REST pública estável.
#       A extração é feita via painel interativo (Qlik) e exportação manual
#       de CSV. Este script documenta o procedimento e lê o arquivo exportado.
#   (D) SICONFI/Tesouro Nacional - API REST aberta - Receita Total (FINBRA/
#       DCA) usada como denominador de PartEmendas
#
# SAÍDA
#   data/processed/emendas_municipios_ba.rds  (painel município x ano)
#
# VERSÃO: corrige o bloco (A) CGU, que usava uma abordagem fundamentalmente
#   equivocada (1 chamada por município-ano, quando o endpoint /emendas não
#   filtra por município). Agora baixa por ANO (10 chamadas externas, não
#   4.170), filtra client-side para municípios da Bahia via `localidadeDoGasto`,
#   corrige o parsing de valores monetários em formato BR, e usa
#   checkpoint/resume por (ano, página) em data/cache/.
################################################################################

# Instalação automática de pacotes faltantes (o script antes assumia que
# todos já estavam instalados - foi assim que "here" apareceu faltando;
# httr2/jsonlite/janitor/stringi teriam o mesmo problema em outra máquina).
pacotes <- c("tidyverse", "httr2", "jsonlite", "here", "janitor", "stringi")

instalar_faltantes <- function(pkgs) {
  faltantes <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(faltantes) > 0) install.packages(faltantes, dependencies = TRUE)
}
instalar_faltantes(pacotes)

library(tidyverse)
library(httr2)
library(jsonlite)
library(here)

municipios_ba <- readRDS(here("data", "processed", "municipios_ba.rds"))
params        <- readRDS(here("data", "processed", "params_projeto.rds"))

if (params$portal_transparencia_key == "") {
  stop("Configure PORTAL_TRANSPARENCIA_API_KEY no .Renviron antes de rodar este script.")
}

anos <- seq(params$ano_inicio, params$ano_fim)

dir.create(here("data", "cache"), recursive = TRUE, showWarnings = FALSE)

## ---------------------------------------------------------------------------
## (A) PORTAL DA TRANSPARÊNCIA (CGU) - emendas parlamentares pagas
## ---------------------------------------------------------------------------
# Documentação: https://api.portaldatransparencia.gov.br/swagger-ui.html
# Endpoint utilizado: /api-de-dados/emendas
#
# DESCOBERTA IMPORTANTE (confirmada empiricamente): este endpoint NÃO filtra
# por município. O parâmetro `codigoIbge` é ignorado - uma chamada com
# codigoIbge=290450 retornou registros de Recife-PE, Toritama-PE, Pancas-ES
# etc., ou seja, sempre a mesma listagem NACIONAL daquele ano, paginada.
# Isso é consistente com scripts públicos que usam este mesmo endpoint
# apenas com `ano` e `pagina` (sem filtro de localidade).
#
# Por isso, a abordagem "1 chamada por (município, ano)" da versão anterior
# estava fundamentalmente errada: além de devolver dados que não
# correspondiam ao município pedido, obrigava a paginar a base nacional
# inteira (até ~200-400+ páginas) para CADA um dos 417 municípios,
# repetindo o mesmo trabalho 417 vezes por ano - daí o travamento.
#
# ABORDAGEM CORRIGIDA:
#   1. Baixa TODAS as páginas de emendas por ANO (10 iterações no nível
#      externo, não 4.170) - endpoint aceita apenas ano + pagina.
#   2. Filtra client-side (no R) os registros cujo `localidadeDoGasto`
#      termina em " - BA" (nível município; registros terminados em
#      "(UF)" são gasto em nível estadual, sem município - ficam de fora).
#   3. Casa o nome do município extraído de `localidadeDoGasto` com a
#      base-mestre do IBGE (normalizando maiúsculas/acentos) para obter
#      o cod_ibge6.
#   4. Corrige o parsing de valores monetários: a API devolve texto no
#      formato brasileiro ("1.412,00"), que `as.numeric()` direto
#      transforma silenciosamente em NA - a versão anterior tinha esse bug.
#
# Cache/checkpoint agora é por (ano, página) em data/cache/, não mais por
# (município, ano) - muito mais compacto e correto.

baixar_pagina_emendas <- function(ano, pagina, api_key) {
  req <- request("https://api.portaldatransparencia.gov.br/api-de-dados/emendas") |>
    req_headers(`chave-api-dados` = api_key) |>
    req_url_query(ano = ano, pagina = pagina) |>
    req_error(is_error = \(resp) FALSE) |>
    req_retry(
      max_tries = 3,
      backoff = ~ 2^.x,
      is_transient = \(resp) resp_status(resp) == 429
    ) |>
    req_timeout(30)

  tryCatch(
    req_perform(req),
    error = function(e) {
      warning(sprintf("Falha de conexão (ano=%s, página=%s): %s",
                      ano, pagina, conditionMessage(e)))
      NULL
    }
  )
}

# Baixa (com cache por página) todas as páginas de um ano. Registra em
# data/cache/erros_cgu.log qualquer combinação ano/página abandonada.
# Retorna list(dados, completo): `completo = FALSE` sinaliza que o ano parou
# por causa de uma falha (rede/429/HTTP), não porque os dados acabaram - isso
# permite que o chamador saiba quais anos vale a pena tentar de novo.
baixar_emendas_ano <- function(ano, api_key, max_paginas = 600, pausa_seg = 0.3,
                               registros_por_pagina = 15, max_ciclos_429 = 3) {
  pagina <- 1
  ciclos_429 <- 0
  resultados <- list()
  completo <- TRUE

  repeat {
    arq_cache <- here("data", "cache", sprintf("cgu_pag_%s_%04d.rds", ano, pagina))

    if (file.exists(arq_cache)) {
      conteudo <- readRDS(arq_cache)
    } else {
      resp <- baixar_pagina_emendas(ano, pagina, api_key)

      if (is.null(resp)) { completo <- FALSE; break }   # rede indisponível mesmo após retries

      if (resp_status(resp) == 429) {
        ciclos_429 <- ciclos_429 + 1
        if (ciclos_429 > max_ciclos_429) {
          cat(sprintf("[%s] 429 persistente ano=%s página=%s - parando aqui.\n",
                      format(Sys.time()), ano, pagina),
              file = here("data", "cache", "erros_cgu.log"), append = TRUE)
          completo <- FALSE
          break
        }
        Sys.sleep(20)
        next
      }
      if (resp_status(resp) >= 400) {
        cat(sprintf("[%s] Erro %s ano=%s página=%s\n",
                    format(Sys.time()), resp_status(resp), ano, pagina),
            file = here("data", "cache", "erros_cgu.log"), append = TRUE)
        completo <- FALSE
        break
      }
      ciclos_429 <- 0

      conteudo <- tryCatch(
        as_tibble(resp_body_json(resp, simplifyVector = TRUE)),
        error = function(e) tibble()
      )
      saveRDS(conteudo, arq_cache)  # checkpoint por página
      Sys.sleep(pausa_seg)
    }

    if (nrow(conteudo) == 0) break   # fim real dos dados (completo = TRUE)
    resultados[[pagina]] <- conteudo

    # Heurística de última página: CONFIRA registros_por_pagina (assumido 15)
    # contra o comportamento real observado antes de confiar cegamente nisso.
    if (nrow(conteudo) < registros_por_pagina) break   # fim real (completo = TRUE)

    pagina <- pagina + 1
    if (pagina > max_paginas) break   # atingiu o teto de segurança - trata como completo
  }

  list(
    dados = if (length(resultados) == 0) tibble() else bind_rows(resultados),
    completo = completo
  )
}

# --- Download com múltiplas passadas automáticas -----------------------
# Uma falha pontual de rede (timeout, DNS) interrompe apenas o ano em que
# ocorreu, sem perder o que já foi baixado (cache por página). Em vez de
# depender de você rodar o script de novo manualmente, repetimos a busca
# automaticamente só para os anos que ficaram incompletos - como as páginas
# já baixadas vêm do cache, cada nova passada é rápida.

max_passadas <- 5
anos_pendentes <- anos
dados_por_ano <- vector("list", length(anos))
names(dados_por_ano) <- as.character(anos)

for (passada in seq_len(max_passadas)) {
  if (length(anos_pendentes) == 0) break

  message(sprintf(">> Passada %d/%d - baixando/retomando %d ano(s): %s",
                  passada, max_passadas, length(anos_pendentes),
                  paste(anos_pendentes, collapse = ", ")))

  ainda_incompletos <- c()
  for (ano in anos_pendentes) {
    resultado <- baixar_emendas_ano(ano, params$portal_transparencia_key)
    dados_por_ano[[as.character(ano)]] <- resultado$dados
    if (!resultado$completo) ainda_incompletos <- c(ainda_incompletos, ano)
  }
  anos_pendentes <- ainda_incompletos
}

if (length(anos_pendentes) > 0) {
  warning(sprintf(
    "Após %d passadas, os seguintes anos continuam incompletos: %s. ",
    max_passadas, paste(anos_pendentes, collapse = ", ")),
    "Os dados parciais desses anos serão usados mesmo assim. Rode o script ",
    "novamente mais tarde (o cache preserva o progresso) ou aumente ",
    "max_passadas se a rede estiver especialmente instável.")
}

emendas_cgu_brasil <- bind_rows(dados_por_ano)

saveRDS(emendas_cgu_brasil, here("data", "raw", "emendas_cgu_brasil_raw.rds"))

if (nrow(emendas_cgu_brasil) == 0) {
  stop("Nenhuma emenda foi baixada em nenhum ano - verifique data/cache/erros_cgu.log ",
       "e a chave de API antes de prosseguir.")
}

# --- Filtragem client-side para a Bahia + casamento de nomes com o IBGE -----

normalizar_nome <- function(x) {
  # de-para para variações ortográficas conhecidas entre fontes (CGU,
  # Transferegov, IBGE) para o mesmo município - aplicado depois de já
  # padronizar maiúsculas/acentos, então cobre qualquer lado que divergir.
  corrigir_nome_conhecido <- function(x) {
    de_para <- c(
      "SANTA TERESINHA" = "SANTA TEREZINHA",
      "LAGEDO DO TABOCAL" = "LAJEDO DO TABOCAL",
      "MUQUEM DO SAO FRANCISCO" = "MUQUEM DE SAO FRANCISCO"
    )
    ifelse(x %in% names(de_para), de_para[x], x)
  }

  x |>
    stringi::stri_trans_general("Latin-ASCII") |>
    str_to_upper() |>
    str_squish() |>
    corrigir_nome_conhecido()
}

emendas_ba_bruto <- emendas_cgu_brasil |>
  janitor::clean_names() |>
  filter(str_detect(str_trim(localidade_do_gasto), "-\\s*BA$")) |>  # exclui "(UF)" e outros estados
  mutate(
    nome_municipio_bruto = str_trim(str_remove(localidade_do_gasto, "-\\s*BA$")),
    nome_norm = normalizar_nome(nome_municipio_bruto)
  )

municipios_norm <- municipios_ba |>
  mutate(nome_norm = normalizar_nome(nome_municipio)) |>
  select(cod_ibge6, nome_norm)

emendas_ba_casado <- emendas_ba_bruto |>
  left_join(municipios_norm, by = "nome_norm")

nao_casados <- emendas_ba_casado |>
  filter(is.na(cod_ibge6)) |>
  distinct(nome_municipio_bruto)

if (nrow(nao_casados) > 0) {
  warning(sprintf(
    "%d nome(s) em localidadeDoGasto não casaram com a base do IBGE: %s. ",
    nrow(nao_casados), paste(nao_casados$nome_municipio_bruto, collapse = "; ")),
    "Essas linhas ficam de fora do agregado - revise grafias/abreviações ",
    "e ajuste normalizar_nome() ou faça um de-para manual se necessário.")
}

# valores vêm como texto em formato BR ("1.412,00") - as.numeric() direto
# retornaria NA silenciosamente (bug presente na versão anterior do script)
parse_valor_br <- function(x) {
  x |>
    str_remove_all("\\.") |>
    str_replace(",", ".") |>
    as.numeric()
}

emendas_cgu <- emendas_ba_casado |>
  filter(!is.na(cod_ibge6)) |>
  mutate(valor_pago = parse_valor_br(valor_pago)) |>
  group_by(cod_ibge6, ano) |>
  summarise(emendas_cgu_pagas = sum(valor_pago, na.rm = TRUE), .groups = "drop")


## ---------------------------------------------------------------------------
## (B) TRANSFEREGOV.BR - transferências especiais ("emendas Pix", Art. 166-A)
## ---------------------------------------------------------------------------
# NOVA FONTE: API pública oficial do Módulo de Transferências Especiais do
# Transferegov (substitui a tentativa anterior via CKAN do dados.gov.br, que
# passou a exigir autenticação sem aviso). Documentação/OpenAPI:
# https://api-publica.transferegov.gestao.gov.br/especiais/openapi.json
# Não requer chave de API.
#
# Endpoints usados:
#   /beneficiarios-especiais?uf_beneficiario=BA -> lista de municípios da BA
#     que já receberam Transferências Especiais, com id_beneficiario.
#   /planos-acao-especiais?id_beneficiario=<id> -> planos de ação (aprox. 1
#     por emenda/beneficiário), com ano_emenda_parlamentar_plano_acao,
#     valor_custeio_plano_acao, valor_investimento_plano_acao e
#     situacao_plano_acao ("CIENTE"/"IMPEDIDO").
#
# APROXIMAÇÃO ASSUMIDA (documentar na dissertação): em Transferência Especial,
# o valor é repassado integralmente ao ente assim que o plano de ação é aceito
# (situacao_plano_acao == "CIENTE") - diferente de convênio, que libera por
# etapas. Por isso tratamos custeio+investimento dos planos CIENTE como
# "valor pago". Para o valor de liquidação/pagamento efetivo documento a
# documento, seria necessário encadear com
# /relatorios-gestao-documento-liquidacao-especiais (não implementado aqui
# por complexidade - avalie se vale a pena para o nível de precisão exigido).
#
# ATENÇÃO: o nome do campo que contém a lista de itens dentro da resposta
# paginada (schema Paginated*Response) não foi confirmado a partir do
# OpenAPI (a seção de components/schemas não veio completa na consulta).
# extrair_itens() abaixo tenta os nomes mais prováveis ("items", "dados",
# "results", "data") - CONFIRA com uma chamada de teste (rode só o trecho de
# diagnóstico abaixo primeiro) antes de disparar o loop completo.

base_transferegov <- "https://api-publica.transferegov.gestao.gov.br/especiais"

extrair_itens <- function(corpo) {
  for (campo in c("items", "dados", "results", "data")) {
    if (!is.null(corpo[[campo]])) return(corpo[[campo]])
  }
  NULL
}

baixar_paginas_transferegov <- function(path, query = list(), tamanho_pagina = 200) {
  pagina <- 1
  resultados <- list()
  repeat {
    req <- request(base_transferegov) |> req_url_path_append(path)
    req <- do.call(req_url_query,
                   c(list(req), query, list(pagina = pagina, tamanho_da_pagina = tamanho_pagina)))
    req <- req |>
      req_error(is_error = \(resp) FALSE) |>
      req_retry(max_tries = 3, backoff = ~ 2^.x,
                is_transient = \(resp) resp_status(resp) == 429) |>
      req_timeout(30)

    resp <- tryCatch(req_perform(req), error = function(e) NULL)
    if (is.null(resp) || resp_status(resp) >= 400) break

    corpo <- tryCatch(resp_body_json(resp, simplifyVector = TRUE), error = function(e) NULL)
    itens <- extrair_itens(corpo)
    if (is.null(itens) || length(itens) == 0) break

    pagina_tbl <- as_tibble(itens)
    if (nrow(pagina_tbl) == 0) break
    resultados[[pagina]] <- pagina_tbl

    if (nrow(pagina_tbl) < tamanho_pagina) break
    pagina <- pagina + 1
  }
  if (length(resultados) == 0) return(tibble())
  bind_rows(resultados)
}

## --- Diagnóstico rápido (rode isto sozinho antes do loop completo) --------
teste_transferegov <- baixar_paginas_transferegov("/beneficiarios-especiais", list(uf_beneficiario = "BA"))
message("---- Diagnóstico Transferegov (beneficiários BA) ----")
message("Linhas retornadas: ", nrow(teste_transferegov))
if (nrow(teste_transferegov) > 0) {
  message("Colunas: ", paste(names(teste_transferegov), collapse = ", "))
} else {
  message("Nenhum item retornado - confira extrair_itens() contra o formato ",
          "real da resposta (rode manualmente req_perform() + resp_body_json() ",
          "em /beneficiarios-especiais?uf_beneficiario=BA e inspecione com str()).")
}
message("------------------------------------------------------")

beneficiarios_ba <- teste_transferegov |> janitor::clean_names()

if (nrow(beneficiarios_ba) == 0) {
  warning("Nenhum beneficiário da BA retornado pela API do Transferegov - ",
          "revise o diagnóstico acima antes de prosseguir.")
  transferegov_especiais <- tibble(cod_ibge6 = character(), ano = integer(),
                                   emendas_pix_pagas = double())
} else {

  # nome_beneficiario vem prefixado ("MUNICIPIO DE GLORIA", "MUNICIPIO DO ...",
  # "MUNICIPIO DA ...") - a base do IBGE não tem esse prefixo ("Glória"), por
  # isso o casamento falhava para quase todos os municípios. Removemos o
  # prefixo antes de normalizar (normalizar_nome() definida no bloco (A)).
  limpar_prefixo_municipio <- function(x) {
    str_replace(x, "^MUNICIPIO\\s+D[AEO]\\s+", "")
  }

  # ("ESTADO DA BAHIA" é o próprio estado, não um município - excluído aqui.)

  beneficiarios_norm <- beneficiarios_ba |>
    filter(nome_beneficiario != "ESTADO DA BAHIA") |>
    mutate(nome_norm = normalizar_nome(limpar_prefixo_municipio(nome_beneficiario)))

  municipios_norm_pix <- municipios_ba |>
    mutate(nome_norm = normalizar_nome(nome_municipio)) |>
    select(cod_ibge6, nome_norm)

  beneficiarios_casado <- beneficiarios_norm |>
    left_join(municipios_norm_pix, by = "nome_norm")

  nao_casados_pix <- beneficiarios_casado |>
    filter(is.na(cod_ibge6)) |>
    distinct(nome_beneficiario)

  if (nrow(nao_casados_pix) > 0) {
    warning(sprintf(
      "%d beneficiário(s) do Transferegov não casaram com a base do IBGE: %s. ",
      nrow(nao_casados_pix), paste(nao_casados_pix$nome_beneficiario, collapse = "; ")),
      "Revise normalizar_nome() ou faça de-para manual se necessário.")
  }

  ids_validos <- beneficiarios_casado |> filter(!is.na(cod_ibge6)) |> distinct(id_beneficiario, cod_ibge6)

  message(sprintf(">> Baixando planos de ação por beneficiário para %d município(s) da BA...",
                  nrow(ids_validos)))

  planos_pix <- map_dfr(
    ids_validos$id_beneficiario,
    ~ baixar_paginas_transferegov("/planos-acao-especiais", list(id_beneficiario = .x)),
    .progress = TRUE
  )

  if (nrow(planos_pix) == 0) {
    warning("Nenhum plano de ação retornado para os beneficiários da BA - ",
            "confira se id_beneficiario é aceito como filtro (teste manualmente ",
            "para 1 id antes de rodar todos).")
    transferegov_especiais <- tibble(cod_ibge6 = character(), ano = integer(),
                                     emendas_pix_pagas = double())
  } else {
    transferegov_especiais <- planos_pix |>
      janitor::clean_names() |>
      filter(situacao_plano_acao == "CIENTE") |>
      left_join(ids_validos, by = "id_beneficiario") |>
      filter(!is.na(cod_ibge6)) |>
      mutate(
        ano = ano_emenda_parlamentar_plano_acao,
        valor_pix = coalesce(valor_custeio_plano_acao, 0) +
                    coalesce(valor_investimento_plano_acao, 0)
      ) |>
      group_by(cod_ibge6, ano) |>
      summarise(emendas_pix_pagas = sum(valor_pix, na.rm = TRUE), .groups = "drop")
  }
}

saveRDS(transferegov_especiais, here("data", "raw", "transferegov_especiais.rds"))


## ---------------------------------------------------------------------------
## (C) SIGA BRASIL (Senado Federal) - extração manual assistida
## ---------------------------------------------------------------------------
# O Siga Brasil (https://www12.senado.leg.br/orcamento/sigabrasil) roda em
# cima do BI "Qlik Sense" e NÃO oferece endpoint REST estável para automação
# completa em R. O procedimento recomendado é:
#
#   1. Acessar o painel "Emendas Parlamentares" do Siga Brasil
#   2. Filtrar: Esfera = Municipal; UF = BA; Ano = 2015:2024
#   3. Exportar a tabela para CSV ("Exportar dados")
#   4. Salvar o arquivo em data/raw/siga_brasil_emendas.csv
#
# O bloco abaixo apenas LÊ e padroniza esse arquivo exportado manualmente,
# servindo como fonte de checagem cruzada (triangulação) com a CGU, já que
# os valores empenhado/liquidado/pago do Siga Brasil e do Portal da
# Transparência nem sempre batem por conta de granularidade e defasagem
# de atualização entre as bases.

caminho_siga <- here("data", "raw", "siga_brasil_emendas.csv")

if (file.exists(caminho_siga)) {
  siga_brasil <- read_csv2(caminho_siga, locale = locale(encoding = "Latin1")) |>
    janitor::clean_names() |>
    filter(uf == "BA") |>
    transmute(
      cod_ibge6 = str_sub(as.character(codigo_ibge_municipio), 1, 6),
      ano = as.integer(ano),
      emendas_siga_pagas = as.numeric(valor_pago)
    ) |>
    group_by(cod_ibge6, ano) |>
    summarise(emendas_siga_pagas = sum(emendas_siga_pagas, na.rm = TRUE), .groups = "drop")
} else {
  message("Arquivo data/raw/siga_brasil_emendas.csv não encontrado. ",
          "Pulei a triangulação com o Siga Brasil (etapa manual - ver comentário acima).")
  siga_brasil <- tibble(cod_ibge6 = character(), ano = integer(),
                        emendas_siga_pagas = double())
}

## ---------------------------------------------------------------------------
## (D) SICONFI (Tesouro Nacional) - Receita Total dos municípios
## ---------------------------------------------------------------------------
# API aberta, sem chave. Documentação:
# https://apidatalake.tesouro.gov.br/docs/siconfi/
# Usamos o relatório RREO (Anexo 1 - Balanço Orçamentário), conta
# "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)", coluna "Até o Bimestre"
# (valor realizado no ano) - confirmado empiricamente via diagnóstico
# (ver bloco abaixo). (ou o DCA/FINBRA quando RREO não estiver disponível
# para o município/ano).
#
# CORREÇÃO: a versão anterior deste bloco filtrava por textos fixos
# ("RECEITAS REALIZADAS", "RECEITAS TOTAIS (III) = (I + II)") nas colunas
# `coluna`/`conta`, supostos sem confirmação contra a resposta real da API -
# e se a formatação exata (acentuação, maiúsculas, parênteses) não bater
# caractere por caractere, o filtro retorna 0 linhas silenciosamente, o que
# gera receita_total = NA em 100% do painel (exatamente o problema relatado).
# A correção abaixo:
#   1. Roda UMA chamada de teste antes do loop completo e imprime os valores
#      reais de `coluna` e `conta` retornados, para diagnosticar o problema
#      antes de gastar tempo baixando os ~4.170 município-anos.
#   2. Usa padrões de busca mais tolerantes (regex parcial, sem exigir a
#      pontuação/parênteses exatos).
#   3. Para com diagnóstico detalhado se o resultado final ficar vazio, em
#      vez de deixar `receita_total`/`PartEmendas` saírem silenciosamente
#      como NA em todo o painel.

baixar_receita_siconfi <- function(cod_ibge7, ano, bimestre = 6) {
  req <- request("https://apidatalake.tesouro.gov.br/ords/siconfi/tt/rreo") |>
    req_url_query(
      an_exercicio = ano,
      nr_periodo = bimestre,        # 6o bimestre = acumulado do ano
      co_tipo_demonstrativo = "RREO",
      id_ente = cod_ibge7
    ) |>
    req_error(is_error = \(resp) FALSE) |>
    req_retry(max_tries = 3, backoff = ~ 2^.x,
              is_transient = \(resp) resp_status(resp) == 429) |>
    req_timeout(90)

  resp <- tryCatch(
    req_perform(req),
    error = function(e) {
      warning(sprintf("Falha de conexão (ibge7=%s, ano=%s): %s",
                      cod_ibge7, ano, conditionMessage(e)))
      NULL
    }
  )
  if (is.null(resp)) return(list(status = NA_integer_, dados = tibble()))
  if (resp_status(resp) >= 400) {
    return(list(status = resp_status(resp), dados = tibble()))
  }

  corpo <- tryCatch(resp_body_json(resp, simplifyVector = TRUE), error = function(e) NULL)
  itens <- if (!is.null(corpo) && !is.null(corpo$items)) as_tibble(corpo$items) else tibble()
  list(status = resp_status(resp), dados = itens)
}

## --- Chamada de teste/diagnóstico (1 município, 1 ano) antes do loop completo

teste_siconfi <- baixar_receita_siconfi(municipios_ba$cod_ibge7[1], anos[1])
message("---- Diagnóstico SICONFI (chamada de teste: ", municipios_ba$nome_municipio[1],
        ", ano ", anos[1], ") ----")
message("Status HTTP: ", teste_siconfi$status)
message("Linhas retornadas: ", nrow(teste_siconfi$dados))
if (nrow(teste_siconfi$dados) > 0) {
  teste_clean <- teste_siconfi$dados |> janitor::clean_names()
  message("Colunas: ", paste(names(teste_clean), collapse = ", "))
  if ("coluna" %in% names(teste_clean)) {
    message("Valores únicos de 'coluna': ", paste(unique(teste_clean$coluna), collapse = " | "))
  }
  if ("conta" %in% names(teste_clean)) {
    message("Amostra de valores de 'conta': ",
            paste(head(unique(teste_clean$conta), 15), collapse = " | "))
  }
} else {
  message("A chamada de teste não retornou nenhuma linha. Isso pode indicar ",
          "que os parâmetros da consulta (an_exercicio/nr_periodo/",
          "co_tipo_demonstrativo/id_ente) não são exatamente os esperados ",
          "pela API atual, ou que este município/ano não tem RREO enviado. ",
          "Confira a documentação vigente em ",
          "https://apidatalake.tesouro.gov.br/docs/siconfi/ antes de prosseguir.")
}
message("--------------------------------------------------------------------")

# Checkpoint/resume por (município, ano), igual ao bloco (A) - uma falha de
# rede no meio do loop não obriga a refazer tudo: rode o script de novo e só
# as combinações que faltarem serão baixadas.
# IMPORTANTE: só grava cache quando a chamada teve uma resposta HTTP de fato
# (status != NA). Se cachear também as falhas de rede (status NA), essas
# combinações ficariam marcadas como "concluídas, sem dados" para sempre -
# um rerun nunca mais tentaria de novo.
baixar_e_cachear_siconfi <- function(cod_ibge7, ano) {
  arq_cache <- here("data", "cache", sprintf("siconfi_%s_%s.rds", cod_ibge7, ano))
  if (file.exists(arq_cache)) return(readRDS(arq_cache))

  Sys.sleep(0.2)
  bruto <- baixar_receita_siconfi(cod_ibge7, ano)

  if (is.na(bruto$status)) {
    # falha de rede - não grava cache, para que um rerun tente de novo
    return(tibble())
  }

  resultado <- bruto$dados |> mutate(cod_ibge7 = cod_ibge7, ano = ano)
  saveRDS(resultado, arq_cache)
  resultado
}

combinacoes_siconfi <- tibble(
  cod_ibge7 = rep(municipios_ba$cod_ibge7, times = length(anos)),
  ano       = rep(anos, each = nrow(municipios_ba))
)

message(sprintf(">> Baixando/lendo cache de receita SICONFI para %d combinações município-ano...",
                nrow(combinacoes_siconfi)))

receita_siconfi_raw <- map2_dfr(
  combinacoes_siconfi$cod_ibge7,
  combinacoes_siconfi$ano,
  baixar_e_cachear_siconfi,
  .progress = TRUE
)

saveRDS(receita_siconfi_raw, here("data", "raw", "receita_siconfi_raw.rds"))

if (nrow(receita_siconfi_raw) == 0) {
  stop(
    "receita_siconfi_raw ficou completamente vazio (0 linhas em todas as ",
    "~", nrow(municipios_ba) * length(anos), " chamadas). O problema está na ",
    "consulta à API do SICONFI em si (parâmetros incorretos, mudança na API, ",
    "ou bloqueio de rede), não no filtro de texto abaixo. Revise o ",
    "'Diagnóstico SICONFI' impresso acima antes de prosseguir - ",
    "PartEmendas ficará NA em todo o painel até isso ser resolvido."
  )
}

receita_siconfi_clean <- receita_siconfi_raw |> janitor::clean_names()
message("Colunas de receita_siconfi_raw (todas as chamadas): ",
        paste(names(receita_siconfi_clean), collapse = ", "))

encontrar_coluna_siconfi <- function(df, padrao, obrigatorio = TRUE) {
  candidatas <- names(df)[str_detect(names(df), padrao)]
  if (length(candidatas) == 0) {
    if (obrigatorio) {
      stop(sprintf("Nenhuma coluna casou com '%s'. Colunas disponíveis: %s",
                   padrao, paste(names(df), collapse = ", ")))
    }
    return(NA_character_)
  }
  candidatas[1]
}
col_coluna_rreo <- encontrar_coluna_siconfi(receita_siconfi_clean, "^coluna$")
col_conta_rreo  <- encontrar_coluna_siconfi(receita_siconfi_clean, "^conta$")
col_valor_rreo  <- encontrar_coluna_siconfi(receita_siconfi_clean, "^valor$")

# CORREÇÃO: o nome de conta usado no filtro ("RECEITAS TOTAIS") nunca existiu
# no RREO Anexo 1 no formato atual da STN - confirmado no diagnóstico: a
# linha de receita agregada se chama "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)
# (I)". Também corrigido antes: filtrar só por `conta` e tomar max(valor)
# pegava TODAS as colunas daquela linha (PREVISÃO INICIAL, PREVISÃO
# ATUALIZADA, No Bimestre, Até o Bimestre, %, SALDO...) - por isso agora
# exigimos também que `coluna` seja "Até o Bimestre" (valor realizado).
# Acentos são removidos antes de comparar (stri_trans_general), para não
# depender de a codificação exata bater caractere por caractere.
sem_acento <- function(x) stringi::stri_trans_general(x, "Latin-ASCII")

receita_total <- receita_siconfi_clean |>
  filter(
    str_detect(toupper(sem_acento(.data[[col_conta_rreo]])),
              "RECEITAS\\s*\\(EXCETO INTRA-ORCAMENTARIAS\\)"),
    str_detect(sem_acento(.data[[col_coluna_rreo]]), "^Ate\\s+o\\s+Bimestre")
  ) |>
  mutate(cod_ibge6 = str_sub(cod_ibge7, 1, 6)) |>
  group_by(cod_ibge6, ano) |>
  summarise(receita_total = suppressWarnings(sum(.data[[col_valor_rreo]], na.rm = TRUE)),
            .groups = "drop") |>
  mutate(receita_total = if_else(is.infinite(receita_total) | receita_total == 0,
                                 NA_real_, receita_total))

# Checagem de sanidade: se sobrar mais de 1 linha por (cod_ibge6, ano) depois
# do filtro, o sum() acima pode estar somando duplicatas (ex.: mais de um
# anexo/demonstrativo retornado pela consulta) em vez de agregar corretamente.
duplicatas_receita <- receita_siconfi_clean |>
  filter(
    str_detect(toupper(sem_acento(.data[[col_conta_rreo]])),
              "RECEITAS\\s*\\(EXCETO INTRA-ORCAMENTARIAS\\)"),
    str_detect(sem_acento(.data[[col_coluna_rreo]]), "^Ate\\s+o\\s+Bimestre")
  ) |>
  mutate(cod_ibge6 = str_sub(cod_ibge7, 1, 6)) |>
  count(cod_ibge6, ano) |>
  filter(n > 1)

if (nrow(duplicatas_receita) > 0) {
  warning(sprintf(
    "%d combinação(ões) de município-ano têm mais de 1 linha casando com ",
    nrow(duplicatas_receita)),
    "'RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)' + 'Até o Bimestre' - revise se ",
    "a soma não está duplicando valores (ex.: mais de um anexo do RREO retornado).")
}

if (nrow(receita_total) == 0 || all(is.na(receita_total$receita_total))) {
  stop(
    "O filtro por 'RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)' + 'Até o Bimestre' ",
    "não encontrou nenhuma linha válida, mesmo com receita_siconfi_raw tendo ",
    "dados. Isso confirma que o texto usado no filtro não bate com os ",
    "valores reais - revise 'Valores únicos de coluna' e 'Amostra de valores ",
    "de conta' impressos no diagnóstico no início deste bloco, e ajuste os ",
    "padrões de str_detect() acima conforme o texto exato observado."
  )
}

n_receita_esperado <- nrow(municipios_ba) * length(anos)
if (nrow(receita_total) < n_receita_esperado * 0.9) {
  warning(sprintf(
    "Apenas %d de ~%d município-anos têm receita_total (%.1f%%). ",
    nrow(receita_total), n_receita_esperado,
    100 * nrow(receita_total) / n_receita_esperado),
    "PartEmendas ficará NA para os município-anos sem receita. Isso pode ",
    "ser normal para municípios/anos sem RREO enviado ao Tesouro, mas se a ",
    "proporção for muito baixa, revise o diagnóstico SICONFI impresso acima.")
}

## ---------------------------------------------------------------------------
## Consolidação final: EmendasPC e PartEmendas
## ---------------------------------------------------------------------------

populacao <- readRDS(here("data", "processed", "populacao_ba.rds"))
# ^ gerado pelo script 05_controles_demograficos_ibge_sei.R - rode-o antes
#   deste bloco final, ou comente as linhas de EmendasPC até lá.

painel_base <- expand_grid(cod_ibge6 = municipios_ba$cod_ibge6, ano = anos)

emendas_final <- painel_base |>
  left_join(emendas_cgu,           by = c("cod_ibge6", "ano")) |>
  left_join(transferegov_especiais, by = c("cod_ibge6", "ano")) |>
  left_join(siga_brasil,           by = c("cod_ibge6", "ano")) |>
  left_join(receita_total,         by = c("cod_ibge6", "ano")) |>
  # populacao_ba.rds (de outro script) também traz cod_ibge7/nome_municipio -
  # removidos aqui antes do join para não colidir com os de municipios_ba
  # logo abaixo (o dplyr resolveria a colisão criando "nome_municipio.x"/
  # ".y" em vez de uma coluna única "nome_municipio").
  left_join(populacao |> select(-any_of(c("cod_ibge7", "nome_municipio"))),
            by = c("cod_ibge6", "ano")) |>
  mutate(
    across(c(emendas_cgu_pagas, emendas_pix_pagas), ~ replace_na(.x, 0)),
    emendas_totais_pagas = emendas_cgu_pagas + emendas_pix_pagas,
    EmendasPC   = emendas_totais_pagas / populacao,
    PartEmendas = emendas_totais_pagas / receita_total
  ) |>
  left_join(municipios_ba, by = "cod_ibge6")

saveRDS(emendas_final, here("data", "processed", "emendas_municipios_ba.rds"))
write_csv(emendas_final, here("data", "processed", "emendas_municipios_ba.csv"))

message("Concluído: data/processed/emendas_municipios_ba.rds (EmendasPC e PartEmendas)")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rex’, ‘covr’, ‘palmerpenguins’, ‘plyr’


here() starts at /content



ERROR: Error: Configure PORTAL_TRANSPARENCIA_API_KEY no .Renviron antes de rodar este script.


In [14]:
PORTAL_TRANSPARENCIA_API_KEY=0ee6728c1324a2e27d865a0c72c6aad1

ERROR: Error in parse(text = input): <text>:1:30: unexpected input
1: PORTAL_TRANSPARENCIA_API_KEY=0ee
                                 ^


In [1]:
if (!requireNamespace("usethis", quietly = TRUE)) install.packages("usethis")
usethis::edit_r_environ("project")

ERROR: [1m[33mError[39m in `proj_set()`:[22m
[1m[22m[31m✖[39m Path [34m/content/[39m does not appear to be inside a project or package.
[36mℹ[39m Read more in the help for `usethis::proj_get()`.


In [2]:
Sys.setenv(PORTAL_TRANSPARENCIA_API_KEY = "0ee6728c1324a2e27d865a0c72c6aad1")

# confirme:
Sys.getenv("PORTAL_TRANSPARENCIA_API_KEY")

[1] "0ee6728c1324a2e27d865a0c72c6aad1"

In [5]:
# se estiver usando o pacote googledrive ou o runtime R do Colab com suporte a drive:
system("mkdir -p /content/drive")
# ou, se o Colab R tiver a função nativa de montagem (varia por imagem):
# googledrive::drive_auth() + cópia manual, ou rodar via reticulate a partir do Python

In [6]:
if (!requireNamespace("googledrive", quietly = TRUE)) install.packages("googledrive")
library(googledrive)

drive_auth()
# Isso imprime uma URL. Abra num navegador, autorize, copie o código de
# autorização que aparece, e cole de volta no console do Colab quando pedido.

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No


Selection: 1


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=36a6c3abf22003591430059762f152ee&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQVhscW9pNmluUHh2TV9OSDUtVE83MWpSb3hCSF96ZWh0cTdvWU0teFotLUtobDh1SUQ5VUd4bkZ6WFpJSjlDYktkNkZodyIsInN0YXRlIjoiMzZhNmMzYWJmMjIwMDM1OTE0MzAwNTk3NjJmMTUyZWUifQ==


In [7]:
################################################################################
# 01_emendas_parlamentares.R
#
# OBJETIVO
#   Construir, para os 417 municípios da Bahia e para cada ano do painel,
#   as variáveis:
#     - EmendasPC   = (emendas individuais + bancada + especiais "Pix" pagas) / população
#     - PartEmendas = emendas pagas / Receita Total do município
#
# FONTES
#   (A) Portal da Transparência (CGU) - API REST oficial, requer chave de API
#       -> emendas parlamentares (individuais, de bancada) pagas por município
#   (B) Transferegov.br - dados abertos (CKAN) - transferências especiais
#       ("emendas Pix", Art. 166-A da CF/88) e convênios/transferências
#       voluntárias oriundas de emendas
#   (C) Siga Brasil (Senado Federal) - NÃO possui API REST pública estável.
#       A extração é feita via painel interativo (Qlik) e exportação manual
#       de CSV. Este script documenta o procedimento e lê o arquivo exportado.
#   (D) SICONFI/Tesouro Nacional - API REST aberta - Receita Total (FINBRA/
#       DCA) usada como denominador de PartEmendas
#
# SAÍDA
#   data/processed/emendas_municipios_ba.rds  (painel município x ano)
#
# VERSÃO: corrige o bloco (A) CGU, que usava uma abordagem fundamentalmente
#   equivocada (1 chamada por município-ano, quando o endpoint /emendas não
#   filtra por município). Agora baixa por ANO (10 chamadas externas, não
#   4.170), filtra client-side para municípios da Bahia via `localidadeDoGasto`,
#   corrige o parsing de valores monetários em formato BR, e usa
#   checkpoint/resume por (ano, página) em data/cache/.
################################################################################

# Instalação automática de pacotes faltantes (o script antes assumia que
# todos já estavam instalados - foi assim que "here" apareceu faltando;
# httr2/jsonlite/janitor/stringi teriam o mesmo problema em outra máquina).
pacotes <- c("tidyverse", "httr2", "jsonlite", "here", "janitor", "stringi")

instalar_faltantes <- function(pkgs) {
  faltantes <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(faltantes) > 0) install.packages(faltantes, dependencies = TRUE)
}
instalar_faltantes(pacotes)

library(tidyverse)
library(httr2)
library(jsonlite)
library(here)

municipios_ba <- readRDS(here("data", "processed", "municipios_ba.rds"))
params        <- readRDS(here("data", "processed", "params_projeto.rds"))

if (params$portal_transparencia_key == "") {
  stop("Configure PORTAL_TRANSPARENCIA_API_KEY no .Renviron antes de rodar este script.")
}

anos <- seq(params$ano_inicio, params$ano_fim)

dir.create(here("data", "cache"), recursive = TRUE, showWarnings = FALSE)

## ---------------------------------------------------------------------------
## (A) PORTAL DA TRANSPARÊNCIA (CGU) - emendas parlamentares pagas
## ---------------------------------------------------------------------------
# Documentação: https://api.portaldatransparencia.gov.br/swagger-ui.html
# Endpoint utilizado: /api-de-dados/emendas
#
# DESCOBERTA IMPORTANTE (confirmada empiricamente): este endpoint NÃO filtra
# por município. O parâmetro `codigoIbge` é ignorado - uma chamada com
# codigoIbge=290450 retornou registros de Recife-PE, Toritama-PE, Pancas-ES
# etc., ou seja, sempre a mesma listagem NACIONAL daquele ano, paginada.
# Isso é consistente com scripts públicos que usam este mesmo endpoint
# apenas com `ano` e `pagina` (sem filtro de localidade).
#
# Por isso, a abordagem "1 chamada por (município, ano)" da versão anterior
# estava fundamentalmente errada: além de devolver dados que não
# correspondiam ao município pedido, obrigava a paginar a base nacional
# inteira (até ~200-400+ páginas) para CADA um dos 417 municípios,
# repetindo o mesmo trabalho 417 vezes por ano - daí o travamento.
#
# ABORDAGEM CORRIGIDA:
#   1. Baixa TODAS as páginas de emendas por ANO (10 iterações no nível
#      externo, não 4.170) - endpoint aceita apenas ano + pagina.
#   2. Filtra client-side (no R) os registros cujo `localidadeDoGasto`
#      termina em " - BA" (nível município; registros terminados em
#      "(UF)" são gasto em nível estadual, sem município - ficam de fora).
#   3. Casa o nome do município extraído de `localidadeDoGasto` com a
#      base-mestre do IBGE (normalizando maiúsculas/acentos) para obter
#      o cod_ibge6.
#   4. Corrige o parsing de valores monetários: a API devolve texto no
#      formato brasileiro ("1.412,00"), que `as.numeric()` direto
#      transforma silenciosamente em NA - a versão anterior tinha esse bug.
#
# Cache/checkpoint agora é por (ano, página) em data/cache/, não mais por
# (município, ano) - muito mais compacto e correto.

baixar_pagina_emendas <- function(ano, pagina, api_key) {
  req <- request("https://api.portaldatransparencia.gov.br/api-de-dados/emendas") |>
    req_headers(`chave-api-dados` = api_key) |>
    req_url_query(ano = ano, pagina = pagina) |>
    req_error(is_error = \(resp) FALSE) |>
    req_retry(
      max_tries = 3,
      backoff = ~ 2^.x,
      is_transient = \(resp) resp_status(resp) == 429
    ) |>
    req_timeout(30)

  tryCatch(
    req_perform(req),
    error = function(e) {
      warning(sprintf("Falha de conexão (ano=%s, página=%s): %s",
                      ano, pagina, conditionMessage(e)))
      NULL
    }
  )
}

# Baixa (com cache por página) todas as páginas de um ano. Registra em
# data/cache/erros_cgu.log qualquer combinação ano/página abandonada.
# Retorna list(dados, completo): `completo = FALSE` sinaliza que o ano parou
# por causa de uma falha (rede/429/HTTP), não porque os dados acabaram - isso
# permite que o chamador saiba quais anos vale a pena tentar de novo.
baixar_emendas_ano <- function(ano, api_key, max_paginas = 600, pausa_seg = 0.3,
                               registros_por_pagina = 15, max_ciclos_429 = 3) {
  pagina <- 1
  ciclos_429 <- 0
  resultados <- list()
  completo <- TRUE

  repeat {
    arq_cache <- here("data", "cache", sprintf("cgu_pag_%s_%04d.rds", ano, pagina))

    if (file.exists(arq_cache)) {
      conteudo <- readRDS(arq_cache)
    } else {
      resp <- baixar_pagina_emendas(ano, pagina, api_key)

      if (is.null(resp)) { completo <- FALSE; break }   # rede indisponível mesmo após retries

      if (resp_status(resp) == 429) {
        ciclos_429 <- ciclos_429 + 1
        if (ciclos_429 > max_ciclos_429) {
          cat(sprintf("[%s] 429 persistente ano=%s página=%s - parando aqui.\n",
                      format(Sys.time()), ano, pagina),
              file = here("data", "cache", "erros_cgu.log"), append = TRUE)
          completo <- FALSE
          break
        }
        Sys.sleep(20)
        next
      }
      if (resp_status(resp) >= 400) {
        cat(sprintf("[%s] Erro %s ano=%s página=%s\n",
                    format(Sys.time()), resp_status(resp), ano, pagina),
            file = here("data", "cache", "erros_cgu.log"), append = TRUE)
        completo <- FALSE
        break
      }
      ciclos_429 <- 0

      conteudo <- tryCatch(
        as_tibble(resp_body_json(resp, simplifyVector = TRUE)),
        error = function(e) tibble()
      )
      saveRDS(conteudo, arq_cache)  # checkpoint por página
      Sys.sleep(pausa_seg)
    }

    if (nrow(conteudo) == 0) break   # fim real dos dados (completo = TRUE)
    resultados[[pagina]] <- conteudo

    # Heurística de última página: CONFIRA registros_por_pagina (assumido 15)
    # contra o comportamento real observado antes de confiar cegamente nisso.
    if (nrow(conteudo) < registros_por_pagina) break   # fim real (completo = TRUE)

    pagina <- pagina + 1
    if (pagina > max_paginas) break   # atingiu o teto de segurança - trata como completo
  }

  list(
    dados = if (length(resultados) == 0) tibble() else bind_rows(resultados),
    completo = completo
  )
}

# --- Download com múltiplas passadas automáticas -----------------------
# Uma falha pontual de rede (timeout, DNS) interrompe apenas o ano em que
# ocorreu, sem perder o que já foi baixado (cache por página). Em vez de
# depender de você rodar o script de novo manualmente, repetimos a busca
# automaticamente só para os anos que ficaram incompletos - como as páginas
# já baixadas vêm do cache, cada nova passada é rápida.

max_passadas <- 5
anos_pendentes <- anos
dados_por_ano <- vector("list", length(anos))
names(dados_por_ano) <- as.character(anos)

for (passada in seq_len(max_passadas)) {
  if (length(anos_pendentes) == 0) break

  message(sprintf(">> Passada %d/%d - baixando/retomando %d ano(s): %s",
                  passada, max_passadas, length(anos_pendentes),
                  paste(anos_pendentes, collapse = ", ")))

  ainda_incompletos <- c()
  for (ano in anos_pendentes) {
    resultado <- baixar_emendas_ano(ano, params$portal_transparencia_key)
    dados_por_ano[[as.character(ano)]] <- resultado$dados
    if (!resultado$completo) ainda_incompletos <- c(ainda_incompletos, ano)
  }
  anos_pendentes <- ainda_incompletos
}

if (length(anos_pendentes) > 0) {
  warning(sprintf(
    "Após %d passadas, os seguintes anos continuam incompletos: %s. ",
    max_passadas, paste(anos_pendentes, collapse = ", ")),
    "Os dados parciais desses anos serão usados mesmo assim. Rode o script ",
    "novamente mais tarde (o cache preserva o progresso) ou aumente ",
    "max_passadas se a rede estiver especialmente instável.")
}

emendas_cgu_brasil <- bind_rows(dados_por_ano)

saveRDS(emendas_cgu_brasil, here("data", "raw", "emendas_cgu_brasil_raw.rds"))

if (nrow(emendas_cgu_brasil) == 0) {
  stop("Nenhuma emenda foi baixada em nenhum ano - verifique data/cache/erros_cgu.log ",
       "e a chave de API antes de prosseguir.")
}

# --- Filtragem client-side para a Bahia + casamento de nomes com o IBGE -----

normalizar_nome <- function(x) {
  # de-para para variações ortográficas conhecidas entre fontes (CGU,
  # Transferegov, IBGE) para o mesmo município - aplicado depois de já
  # padronizar maiúsculas/acentos, então cobre qualquer lado que divergir.
  corrigir_nome_conhecido <- function(x) {
    de_para <- c(
      "SANTA TERESINHA" = "SANTA TEREZINHA",
      "LAGEDO DO TABOCAL" = "LAJEDO DO TABOCAL",
      "MUQUEM DO SAO FRANCISCO" = "MUQUEM DE SAO FRANCISCO"
    )
    ifelse(x %in% names(de_para), de_para[x], x)
  }

  x |>
    stringi::stri_trans_general("Latin-ASCII") |>
    str_to_upper() |>
    str_squish() |>
    corrigir_nome_conhecido()
}

emendas_ba_bruto <- emendas_cgu_brasil |>
  janitor::clean_names() |>
  filter(str_detect(str_trim(localidade_do_gasto), "-\\s*BA$")) |>  # exclui "(UF)" e outros estados
  mutate(
    nome_municipio_bruto = str_trim(str_remove(localidade_do_gasto, "-\\s*BA$")),
    nome_norm = normalizar_nome(nome_municipio_bruto)
  )

municipios_norm <- municipios_ba |>
  mutate(nome_norm = normalizar_nome(nome_municipio)) |>
  select(cod_ibge6, nome_norm)

emendas_ba_casado <- emendas_ba_bruto |>
  left_join(municipios_norm, by = "nome_norm")

nao_casados <- emendas_ba_casado |>
  filter(is.na(cod_ibge6)) |>
  distinct(nome_municipio_bruto)

if (nrow(nao_casados) > 0) {
  warning(sprintf(
    "%d nome(s) em localidadeDoGasto não casaram com a base do IBGE: %s. ",
    nrow(nao_casados), paste(nao_casados$nome_municipio_bruto, collapse = "; ")),
    "Essas linhas ficam de fora do agregado - revise grafias/abreviações ",
    "e ajuste normalizar_nome() ou faça um de-para manual se necessário.")
}

# valores vêm como texto em formato BR ("1.412,00") - as.numeric() direto
# retornaria NA silenciosamente (bug presente na versão anterior do script)
parse_valor_br <- function(x) {
  x |>
    str_remove_all("\\.") |>
    str_replace(",", ".") |>
    as.numeric()
}

emendas_cgu <- emendas_ba_casado |>
  filter(!is.na(cod_ibge6)) |>
  mutate(valor_pago = parse_valor_br(valor_pago)) |>
  group_by(cod_ibge6, ano) |>
  summarise(emendas_cgu_pagas = sum(valor_pago, na.rm = TRUE), .groups = "drop")


## ---------------------------------------------------------------------------
## (B) TRANSFEREGOV.BR - transferências especiais ("emendas Pix", Art. 166-A)
## ---------------------------------------------------------------------------
# NOVA FONTE: API pública oficial do Módulo de Transferências Especiais do
# Transferegov (substitui a tentativa anterior via CKAN do dados.gov.br, que
# passou a exigir autenticação sem aviso). Documentação/OpenAPI:
# https://api-publica.transferegov.gestao.gov.br/especiais/openapi.json
# Não requer chave de API.
#
# Endpoints usados:
#   /beneficiarios-especiais?uf_beneficiario=BA -> lista de municípios da BA
#     que já receberam Transferências Especiais, com id_beneficiario.
#   /planos-acao-especiais?id_beneficiario=<id> -> planos de ação (aprox. 1
#     por emenda/beneficiário), com ano_emenda_parlamentar_plano_acao,
#     valor_custeio_plano_acao, valor_investimento_plano_acao e
#     situacao_plano_acao ("CIENTE"/"IMPEDIDO").
#
# APROXIMAÇÃO ASSUMIDA (documentar na dissertação): em Transferência Especial,
# o valor é repassado integralmente ao ente assim que o plano de ação é aceito
# (situacao_plano_acao == "CIENTE") - diferente de convênio, que libera por
# etapas. Por isso tratamos custeio+investimento dos planos CIENTE como
# "valor pago". Para o valor de liquidação/pagamento efetivo documento a
# documento, seria necessário encadear com
# /relatorios-gestao-documento-liquidacao-especiais (não implementado aqui
# por complexidade - avalie se vale a pena para o nível de precisão exigido).
#
# ATENÇÃO: o nome do campo que contém a lista de itens dentro da resposta
# paginada (schema Paginated*Response) não foi confirmado a partir do
# OpenAPI (a seção de components/schemas não veio completa na consulta).
# extrair_itens() abaixo tenta os nomes mais prováveis ("items", "dados",
# "results", "data") - CONFIRA com uma chamada de teste (rode só o trecho de
# diagnóstico abaixo primeiro) antes de disparar o loop completo.

base_transferegov <- "https://api-publica.transferegov.gestao.gov.br/especiais"

extrair_itens <- function(corpo) {
  for (campo in c("items", "dados", "results", "data")) {
    if (!is.null(corpo[[campo]])) return(corpo[[campo]])
  }
  NULL
}

baixar_paginas_transferegov <- function(path, query = list(), tamanho_pagina = 200) {
  pagina <- 1
  resultados <- list()
  repeat {
    req <- request(base_transferegov) |> req_url_path_append(path)
    req <- do.call(req_url_query,
                   c(list(req), query, list(pagina = pagina, tamanho_da_pagina = tamanho_pagina)))
    req <- req |>
      req_error(is_error = \(resp) FALSE) |>
      req_retry(max_tries = 3, backoff = ~ 2^.x,
                is_transient = \(resp) resp_status(resp) == 429) |>
      req_timeout(30)

    resp <- tryCatch(req_perform(req), error = function(e) NULL)
    if (is.null(resp) || resp_status(resp) >= 400) break

    corpo <- tryCatch(resp_body_json(resp, simplifyVector = TRUE), error = function(e) NULL)
    itens <- extrair_itens(corpo)
    if (is.null(itens) || length(itens) == 0) break

    pagina_tbl <- as_tibble(itens)
    if (nrow(pagina_tbl) == 0) break
    resultados[[pagina]] <- pagina_tbl

    if (nrow(pagina_tbl) < tamanho_pagina) break
    pagina <- pagina + 1
  }
  if (length(resultados) == 0) return(tibble())
  bind_rows(resultados)
}

## --- Diagnóstico rápido (rode isto sozinho antes do loop completo) --------
teste_transferegov <- baixar_paginas_transferegov("/beneficiarios-especiais", list(uf_beneficiario = "BA"))
message("---- Diagnóstico Transferegov (beneficiários BA) ----")
message("Linhas retornadas: ", nrow(teste_transferegov))
if (nrow(teste_transferegov) > 0) {
  message("Colunas: ", paste(names(teste_transferegov), collapse = ", "))
} else {
  message("Nenhum item retornado - confira extrair_itens() contra o formato ",
          "real da resposta (rode manualmente req_perform() + resp_body_json() ",
          "em /beneficiarios-especiais?uf_beneficiario=BA e inspecione com str()).")
}
message("------------------------------------------------------")

beneficiarios_ba <- teste_transferegov |> janitor::clean_names()

if (nrow(beneficiarios_ba) == 0) {
  warning("Nenhum beneficiário da BA retornado pela API do Transferegov - ",
          "revise o diagnóstico acima antes de prosseguir.")
  transferegov_especiais <- tibble(cod_ibge6 = character(), ano = integer(),
                                   emendas_pix_pagas = double())
} else {

  # nome_beneficiario vem prefixado ("MUNICIPIO DE GLORIA", "MUNICIPIO DO ...",
  # "MUNICIPIO DA ...") - a base do IBGE não tem esse prefixo ("Glória"), por
  # isso o casamento falhava para quase todos os municípios. Removemos o
  # prefixo antes de normalizar (normalizar_nome() definida no bloco (A)).
  limpar_prefixo_municipio <- function(x) {
    str_replace(x, "^MUNICIPIO\\s+D[AEO]\\s+", "")
  }

  # ("ESTADO DA BAHIA" é o próprio estado, não um município - excluído aqui.)

  beneficiarios_norm <- beneficiarios_ba |>
    filter(nome_beneficiario != "ESTADO DA BAHIA") |>
    mutate(nome_norm = normalizar_nome(limpar_prefixo_municipio(nome_beneficiario)))

  municipios_norm_pix <- municipios_ba |>
    mutate(nome_norm = normalizar_nome(nome_municipio)) |>
    select(cod_ibge6, nome_norm)

  beneficiarios_casado <- beneficiarios_norm |>
    left_join(municipios_norm_pix, by = "nome_norm")

  nao_casados_pix <- beneficiarios_casado |>
    filter(is.na(cod_ibge6)) |>
    distinct(nome_beneficiario)

  if (nrow(nao_casados_pix) > 0) {
    warning(sprintf(
      "%d beneficiário(s) do Transferegov não casaram com a base do IBGE: %s. ",
      nrow(nao_casados_pix), paste(nao_casados_pix$nome_beneficiario, collapse = "; ")),
      "Revise normalizar_nome() ou faça de-para manual se necessário.")
  }

  ids_validos <- beneficiarios_casado |> filter(!is.na(cod_ibge6)) |> distinct(id_beneficiario, cod_ibge6)

  message(sprintf(">> Baixando planos de ação por beneficiário para %d município(s) da BA...",
                  nrow(ids_validos)))

  planos_pix <- map_dfr(
    ids_validos$id_beneficiario,
    ~ baixar_paginas_transferegov("/planos-acao-especiais", list(id_beneficiario = .x)),
    .progress = TRUE
  )

  if (nrow(planos_pix) == 0) {
    warning("Nenhum plano de ação retornado para os beneficiários da BA - ",
            "confira se id_beneficiario é aceito como filtro (teste manualmente ",
            "para 1 id antes de rodar todos).")
    transferegov_especiais <- tibble(cod_ibge6 = character(), ano = integer(),
                                     emendas_pix_pagas = double())
  } else {
    transferegov_especiais <- planos_pix |>
      janitor::clean_names() |>
      filter(situacao_plano_acao == "CIENTE") |>
      left_join(ids_validos, by = "id_beneficiario") |>
      filter(!is.na(cod_ibge6)) |>
      mutate(
        ano = ano_emenda_parlamentar_plano_acao,
        valor_pix = coalesce(valor_custeio_plano_acao, 0) +
                    coalesce(valor_investimento_plano_acao, 0)
      ) |>
      group_by(cod_ibge6, ano) |>
      summarise(emendas_pix_pagas = sum(valor_pix, na.rm = TRUE), .groups = "drop")
  }
}

saveRDS(transferegov_especiais, here("data", "raw", "transferegov_especiais.rds"))


## ---------------------------------------------------------------------------
## (C) SIGA BRASIL (Senado Federal) - extração manual assistida
## ---------------------------------------------------------------------------
# O Siga Brasil (https://www12.senado.leg.br/orcamento/sigabrasil) roda em
# cima do BI "Qlik Sense" e NÃO oferece endpoint REST estável para automação
# completa em R. O procedimento recomendado é:
#
#   1. Acessar o painel "Emendas Parlamentares" do Siga Brasil
#   2. Filtrar: Esfera = Municipal; UF = BA; Ano = 2015:2024
#   3. Exportar a tabela para CSV ("Exportar dados")
#   4. Salvar o arquivo em data/raw/siga_brasil_emendas.csv
#
# O bloco abaixo apenas LÊ e padroniza esse arquivo exportado manualmente,
# servindo como fonte de checagem cruzada (triangulação) com a CGU, já que
# os valores empenhado/liquidado/pago do Siga Brasil e do Portal da
# Transparência nem sempre batem por conta de granularidade e defasagem
# de atualização entre as bases.

caminho_siga <- here("data", "raw", "siga_brasil_emendas.csv")

if (file.exists(caminho_siga)) {
  siga_brasil <- read_csv2(caminho_siga, locale = locale(encoding = "Latin1")) |>
    janitor::clean_names() |>
    filter(uf == "BA") |>
    transmute(
      cod_ibge6 = str_sub(as.character(codigo_ibge_municipio), 1, 6),
      ano = as.integer(ano),
      emendas_siga_pagas = as.numeric(valor_pago)
    ) |>
    group_by(cod_ibge6, ano) |>
    summarise(emendas_siga_pagas = sum(emendas_siga_pagas, na.rm = TRUE), .groups = "drop")
} else {
  message("Arquivo data/raw/siga_brasil_emendas.csv não encontrado. ",
          "Pulei a triangulação com o Siga Brasil (etapa manual - ver comentário acima).")
  siga_brasil <- tibble(cod_ibge6 = character(), ano = integer(),
                        emendas_siga_pagas = double())
}

## ---------------------------------------------------------------------------
## (D) SICONFI (Tesouro Nacional) - Receita Total dos municípios
## ---------------------------------------------------------------------------
# API aberta, sem chave. Documentação:
# https://apidatalake.tesouro.gov.br/docs/siconfi/
# Usamos o relatório RREO (Anexo 1 - Balanço Orçamentário), conta
# "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)", coluna "Até o Bimestre"
# (valor realizado no ano) - confirmado empiricamente via diagnóstico
# (ver bloco abaixo). (ou o DCA/FINBRA quando RREO não estiver disponível
# para o município/ano).
#
# CORREÇÃO: a versão anterior deste bloco filtrava por textos fixos
# ("RECEITAS REALIZADAS", "RECEITAS TOTAIS (III) = (I + II)") nas colunas
# `coluna`/`conta`, supostos sem confirmação contra a resposta real da API -
# e se a formatação exata (acentuação, maiúsculas, parênteses) não bater
# caractere por caractere, o filtro retorna 0 linhas silenciosamente, o que
# gera receita_total = NA em 100% do painel (exatamente o problema relatado).
# A correção abaixo:
#   1. Roda UMA chamada de teste antes do loop completo e imprime os valores
#      reais de `coluna` e `conta` retornados, para diagnosticar o problema
#      antes de gastar tempo baixando os ~4.170 município-anos.
#   2. Usa padrões de busca mais tolerantes (regex parcial, sem exigir a
#      pontuação/parênteses exatos).
#   3. Para com diagnóstico detalhado se o resultado final ficar vazio, em
#      vez de deixar `receita_total`/`PartEmendas` saírem silenciosamente
#      como NA em todo o painel.

baixar_receita_siconfi <- function(cod_ibge7, ano, bimestre = 6) {
  req <- request("https://apidatalake.tesouro.gov.br/ords/siconfi/tt/rreo") |>
    req_url_query(
      an_exercicio = ano,
      nr_periodo = bimestre,        # 6o bimestre = acumulado do ano
      co_tipo_demonstrativo = "RREO",
      id_ente = cod_ibge7
    ) |>
    req_error(is_error = \(resp) FALSE) |>
    req_retry(max_tries = 3, backoff = ~ 2^.x,
              is_transient = \(resp) resp_status(resp) == 429) |>
    req_timeout(90)

  resp <- tryCatch(
    req_perform(req),
    error = function(e) {
      warning(sprintf("Falha de conexão (ibge7=%s, ano=%s): %s",
                      cod_ibge7, ano, conditionMessage(e)))
      NULL
    }
  )
  if (is.null(resp)) return(list(status = NA_integer_, dados = tibble()))
  if (resp_status(resp) >= 400) {
    return(list(status = resp_status(resp), dados = tibble()))
  }

  corpo <- tryCatch(resp_body_json(resp, simplifyVector = TRUE), error = function(e) NULL)
  itens <- if (!is.null(corpo) && !is.null(corpo$items)) as_tibble(corpo$items) else tibble()
  list(status = resp_status(resp), dados = itens)
}

## --- Chamada de teste/diagnóstico (1 município, 1 ano) antes do loop completo

teste_siconfi <- baixar_receita_siconfi(municipios_ba$cod_ibge7[1], anos[1])
message("---- Diagnóstico SICONFI (chamada de teste: ", municipios_ba$nome_municipio[1],
        ", ano ", anos[1], ") ----")
message("Status HTTP: ", teste_siconfi$status)
message("Linhas retornadas: ", nrow(teste_siconfi$dados))
if (nrow(teste_siconfi$dados) > 0) {
  teste_clean <- teste_siconfi$dados |> janitor::clean_names()
  message("Colunas: ", paste(names(teste_clean), collapse = ", "))
  if ("coluna" %in% names(teste_clean)) {
    message("Valores únicos de 'coluna': ", paste(unique(teste_clean$coluna), collapse = " | "))
  }
  if ("conta" %in% names(teste_clean)) {
    message("Amostra de valores de 'conta': ",
            paste(head(unique(teste_clean$conta), 15), collapse = " | "))
  }
} else {
  message("A chamada de teste não retornou nenhuma linha. Isso pode indicar ",
          "que os parâmetros da consulta (an_exercicio/nr_periodo/",
          "co_tipo_demonstrativo/id_ente) não são exatamente os esperados ",
          "pela API atual, ou que este município/ano não tem RREO enviado. ",
          "Confira a documentação vigente em ",
          "https://apidatalake.tesouro.gov.br/docs/siconfi/ antes de prosseguir.")
}
message("--------------------------------------------------------------------")

# Checkpoint/resume por (município, ano), igual ao bloco (A) - uma falha de
# rede no meio do loop não obriga a refazer tudo: rode o script de novo e só
# as combinações que faltarem serão baixadas.
# IMPORTANTE: só grava cache quando a chamada teve uma resposta HTTP de fato
# (status != NA). Se cachear também as falhas de rede (status NA), essas
# combinações ficariam marcadas como "concluídas, sem dados" para sempre -
# um rerun nunca mais tentaria de novo.
baixar_e_cachear_siconfi <- function(cod_ibge7, ano) {
  arq_cache <- here("data", "cache", sprintf("siconfi_%s_%s.rds", cod_ibge7, ano))
  if (file.exists(arq_cache)) return(readRDS(arq_cache))

  Sys.sleep(0.2)
  bruto <- baixar_receita_siconfi(cod_ibge7, ano)

  if (is.na(bruto$status)) {
    # falha de rede - não grava cache, para que um rerun tente de novo
    return(tibble())
  }

  resultado <- bruto$dados |> mutate(cod_ibge7 = cod_ibge7, ano = ano)
  saveRDS(resultado, arq_cache)
  resultado
}

combinacoes_siconfi <- tibble(
  cod_ibge7 = rep(municipios_ba$cod_ibge7, times = length(anos)),
  ano       = rep(anos, each = nrow(municipios_ba))
)

message(sprintf(">> Baixando/lendo cache de receita SICONFI para %d combinações município-ano...",
                nrow(combinacoes_siconfi)))

receita_siconfi_raw <- map2_dfr(
  combinacoes_siconfi$cod_ibge7,
  combinacoes_siconfi$ano,
  baixar_e_cachear_siconfi,
  .progress = TRUE
)

saveRDS(receita_siconfi_raw, here("data", "raw", "receita_siconfi_raw.rds"))

if (nrow(receita_siconfi_raw) == 0) {
  stop(
    "receita_siconfi_raw ficou completamente vazio (0 linhas em todas as ",
    "~", nrow(municipios_ba) * length(anos), " chamadas). O problema está na ",
    "consulta à API do SICONFI em si (parâmetros incorretos, mudança na API, ",
    "ou bloqueio de rede), não no filtro de texto abaixo. Revise o ",
    "'Diagnóstico SICONFI' impresso acima antes de prosseguir - ",
    "PartEmendas ficará NA em todo o painel até isso ser resolvido."
  )
}

receita_siconfi_clean <- receita_siconfi_raw |> janitor::clean_names()
message("Colunas de receita_siconfi_raw (todas as chamadas): ",
        paste(names(receita_siconfi_clean), collapse = ", "))

encontrar_coluna_siconfi <- function(df, padrao, obrigatorio = TRUE) {
  candidatas <- names(df)[str_detect(names(df), padrao)]
  if (length(candidatas) == 0) {
    if (obrigatorio) {
      stop(sprintf("Nenhuma coluna casou com '%s'. Colunas disponíveis: %s",
                   padrao, paste(names(df), collapse = ", ")))
    }
    return(NA_character_)
  }
  candidatas[1]
}
col_coluna_rreo <- encontrar_coluna_siconfi(receita_siconfi_clean, "^coluna$")
col_conta_rreo  <- encontrar_coluna_siconfi(receita_siconfi_clean, "^conta$")
col_valor_rreo  <- encontrar_coluna_siconfi(receita_siconfi_clean, "^valor$")

# CORREÇÃO: o nome de conta usado no filtro ("RECEITAS TOTAIS") nunca existiu
# no RREO Anexo 1 no formato atual da STN - confirmado no diagnóstico: a
# linha de receita agregada se chama "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)
# (I)". Também corrigido antes: filtrar só por `conta` e tomar max(valor)
# pegava TODAS as colunas daquela linha (PREVISÃO INICIAL, PREVISÃO
# ATUALIZADA, No Bimestre, Até o Bimestre, %, SALDO...) - por isso agora
# exigimos também que `coluna` seja "Até o Bimestre" (valor realizado).
# Acentos são removidos antes de comparar (stri_trans_general), para não
# depender de a codificação exata bater caractere por caractere.
sem_acento <- function(x) stringi::stri_trans_general(x, "Latin-ASCII")

receita_total <- receita_siconfi_clean |>
  filter(
    str_detect(toupper(sem_acento(.data[[col_conta_rreo]])),
              "RECEITAS\\s*\\(EXCETO INTRA-ORCAMENTARIAS\\)"),
    str_detect(sem_acento(.data[[col_coluna_rreo]]), "^Ate\\s+o\\s+Bimestre")
  ) |>
  mutate(cod_ibge6 = str_sub(cod_ibge7, 1, 6)) |>
  group_by(cod_ibge6, ano) |>
  summarise(receita_total = suppressWarnings(sum(.data[[col_valor_rreo]], na.rm = TRUE)),
            .groups = "drop") |>
  mutate(receita_total = if_else(is.infinite(receita_total) | receita_total == 0,
                                 NA_real_, receita_total))

# Checagem de sanidade: se sobrar mais de 1 linha por (cod_ibge6, ano) depois
# do filtro, o sum() acima pode estar somando duplicatas (ex.: mais de um
# anexo/demonstrativo retornado pela consulta) em vez de agregar corretamente.
duplicatas_receita <- receita_siconfi_clean |>
  filter(
    str_detect(toupper(sem_acento(.data[[col_conta_rreo]])),
              "RECEITAS\\s*\\(EXCETO INTRA-ORCAMENTARIAS\\)"),
    str_detect(sem_acento(.data[[col_coluna_rreo]]), "^Ate\\s+o\\s+Bimestre")
  ) |>
  mutate(cod_ibge6 = str_sub(cod_ibge7, 1, 6)) |>
  count(cod_ibge6, ano) |>
  filter(n > 1)

if (nrow(duplicatas_receita) > 0) {
  warning(sprintf(
    "%d combinação(ões) de município-ano têm mais de 1 linha casando com ",
    nrow(duplicatas_receita)),
    "'RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)' + 'Até o Bimestre' - revise se ",
    "a soma não está duplicando valores (ex.: mais de um anexo do RREO retornado).")
}

if (nrow(receita_total) == 0 || all(is.na(receita_total$receita_total))) {
  stop(
    "O filtro por 'RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS)' + 'Até o Bimestre' ",
    "não encontrou nenhuma linha válida, mesmo com receita_siconfi_raw tendo ",
    "dados. Isso confirma que o texto usado no filtro não bate com os ",
    "valores reais - revise 'Valores únicos de coluna' e 'Amostra de valores ",
    "de conta' impressos no diagnóstico no início deste bloco, e ajuste os ",
    "padrões de str_detect() acima conforme o texto exato observado."
  )
}

n_receita_esperado <- nrow(municipios_ba) * length(anos)
if (nrow(receita_total) < n_receita_esperado * 0.9) {
  warning(sprintf(
    "Apenas %d de ~%d município-anos têm receita_total (%.1f%%). ",
    nrow(receita_total), n_receita_esperado,
    100 * nrow(receita_total) / n_receita_esperado),
    "PartEmendas ficará NA para os município-anos sem receita. Isso pode ",
    "ser normal para municípios/anos sem RREO enviado ao Tesouro, mas se a ",
    "proporção for muito baixa, revise o diagnóstico SICONFI impresso acima.")
}

## ---------------------------------------------------------------------------
## Consolidação final: EmendasPC e PartEmendas
## ---------------------------------------------------------------------------

populacao <- readRDS(here("data", "processed", "populacao_ba.rds"))
# ^ gerado pelo script 05_controles_demograficos_ibge_sei.R - rode-o antes
#   deste bloco final, ou comente as linhas de EmendasPC até lá.

painel_base <- expand_grid(cod_ibge6 = municipios_ba$cod_ibge6, ano = anos)

emendas_final <- painel_base |>
  left_join(emendas_cgu,           by = c("cod_ibge6", "ano")) |>
  left_join(transferegov_especiais, by = c("cod_ibge6", "ano")) |>
  left_join(siga_brasil,           by = c("cod_ibge6", "ano")) |>
  left_join(receita_total,         by = c("cod_ibge6", "ano")) |>
  # populacao_ba.rds (de outro script) também traz cod_ibge7/nome_municipio -
  # removidos aqui antes do join para não colidir com os de municipios_ba
  # logo abaixo (o dplyr resolveria a colisão criando "nome_municipio.x"/
  # ".y" em vez de uma coluna única "nome_municipio").
  left_join(populacao |> select(-any_of(c("cod_ibge7", "nome_municipio"))),
            by = c("cod_ibge6", "ano")) |>
  mutate(
    across(c(emendas_cgu_pagas, emendas_pix_pagas), ~ replace_na(.x, 0)),
    emendas_totais_pagas = emendas_cgu_pagas + emendas_pix_pagas,
    EmendasPC   = emendas_totais_pagas / populacao,
    PartEmendas = emendas_totais_pagas / receita_total
  ) |>
  left_join(municipios_ba, by = "cod_ibge6")

saveRDS(emendas_final, here("data", "processed", "emendas_municipios_ba.rds"))
write_csv(emendas_final, here("data", "processed", "emendas_municipios_ba.csv"))

message("Concluído: data/processed/emendas_municipios_ba.rds (EmendasPC e PartEmendas)")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rex’, ‘covr’, ‘palmerpenguins’, ‘plyr’


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ purrr     1.2.2
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter()  masks stats::filter()
✖ purrr::flatten() masks jsonlite::flatten()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /content

>> Passada 1/5 - baixando/retomando 10 ano(s): 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024

---- Diagnóstico Transferegov (beneficiários BA) ----

Linhas retornadas: 410

Colunas: id_beneficiario, uf_beneficiario, nome_beneficiario, cnpj_beneficiario, id_ente

-------------------------------

ERROR: Error in gzfile(file, "rb"): cannot open the connection
